In [192]:
pip install thefuzz

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


#PASO 1. Cargar y revisar los libros

In [193]:
import pandas as pd 
from thefuzz import fuzz
# O import Levenshtein / thefuzz

##Carga del catálogo (libros)

In [194]:
ruta_catalogo = "./catalogo_tp3.jsonl" 
catalogo_libros = pd.read_json(ruta_catalogo, lines=True)

In [195]:
catalogo_libros.head(5)

,source_url,slug,title,authors,genres,seed_genre,series,series_number,cover_url,synopsis,synopsis_chars,scraped_at
0,https://ww3.lectulandia.com/book/de-la-caverna...,de-la-caverna-al-cosmos,De la caverna al cosmos,[Eudald Carbonell],"[Arqueología, Divulgación, Ensayo]",arqueologia,NaN,NaN,https://assets.lectulandia.com/b/ab/Eudald%20C...,La Humanidad ha entrado en una nueva fase evol...,998,2026-10-17 23:11:56+00:00
1,https://ww3.lectulandia.com/book/quienes-somos...,quienes-somos-y-como-hemos-llegado-hasta-aqui,Quiénes somos y cómo hemos llegado hasta aquí,[David Reich],"[Arqueología, Biología, Ciencias naturales, En...",arqueologia,NaN,NaN,https://assets.lectulandia.com/b/ab/David%20Re...,La extracción de ADN de los huesos de nuestros...,1285,2026-10-17 23:11:58+00:00
2,https://ww3.lectulandia.com/book/teotihuacan/,teotihuacan,Teotihuacan,[Eduardo Matos Moctezuma],"[Arqueología, Arte, Ciencias sociales, Ensayo,...",arqueologia,NaN,NaN,https://assets.lectulandia.com/b/ab/Eduardo%20...,Teotihuacan es la primera gran ciudad del cent...,1070,2026-10-17 23:12:01+00:00
3,https://ww3.lectulandia.com/book/tiro-y-las-co...,tiro-y-las-colonias-fenicias-de-occidente,Tiro y las colonias fenicias de Occidente,[M.ª Eugenia Aubet],"[Arqueología, Economía, Ensayo, Historia, Polí...",arqueologia,NaN,NaN,https://assets.lectulandia.com/b/ab/M%20Eugeni...,Entre los siglos VIII y VI a. C. los fenicios ...,1875,2026-10-17 23:12:03+00:00
4,https://ww3.lectulandia.com/book/la-maldicion-...,la-maldicion-del-hombre-mono,La maldición del hombre mono,[Emiliano Bruner],"[Arqueología, Ensayo, Psicología, Sociología]",arqueologia,NaN,NaN,https://assets.lectulandia.com/b/ab/Emiliano%2...,Las raíces evolutivas del sufrimiento humano\n...,1613,2026-10-17 23:12:05+00:00


In [196]:
cantidad_inicial = len(catalogo_libros)
print(f'Cantidad de libros inicial cargados: {cantidad_inicial}')

Cantidad de libros inicial cargados: 8747


In [197]:
libros = catalogo_libros[['slug', 'title', 'authors', 'synopsis']].copy()
#libros.head(5)

##Eliminación de registros sin sinopsis

In [198]:
#Linea con sinopsis nula o vacía
sinopsis_nula =libros[
    libros['synopsis'].isnull() |
    (libros['synopsis'].str.strip() == '')
]
sinopsis_nula

,slug,title,authors,synopsis
7969,calvario,Calvario,[Alejandro Sawa],NaN


In [199]:
#Libros sin sinopsis nula o vacía
libros = libros[
    libros['synopsis'].notnull() &
    (libros['synopsis'].str.strip() != '')
]

In [200]:
print(f"Cantidad de libros con sinopsis: {len(libros)}")

Cantidad de libros con sinopsis: 8746


##Eliminación de duplicados en slug

In [201]:
libros['slug'].duplicated().sum()

np.int64(0)

Como no hay duplicados no se procede a su eliminación 

In [202]:
eliminados = {'sin sinopsis': cantidad_inicial - len(libros), 'slug duplicado': 0}
eliminados

{'sin sinopsis': 1, 'slug duplicado': 0}

In [203]:
cantidad_total = len(libros)
cantidad_total

8746

##Normalización

In [204]:
columnas_texto = ['slug', 'title', 'synopsis']


filas_a_normalizar = libros[
    libros[columnas_texto].apply(
        lambda columna: columna.str.contains(r'\s{2,}', na=False)
    ).any(axis=1)
]

filas_a_normalizar[columnas_texto]


,slug,title,synopsis
433,cuentos-para-no-tan-ninos,Cuentos para (no tan) niños,Los Cuentos Para (No Tan) Niños son un centena...
440,la-dama-de-urtubi,La Dama de Urtubi,Se trata de una ficción aunque hay numerosos \...
604,agilmente,Agilmente,"""Hoy la neurociencia es clara: el cerebro apre..."
773,arcanum-ilimitado,Arcanum ilimitado,La esperada antología de relatos y novelas cor...
791,pacto-de-sangre,Pacto de sangre,"Desde hace siglos, vampiros y licántropos han ..."
1051,nunca-digas-nunca,Nunca digas nunca,La vida de Jacq da un giro trágico e imprevist...
1356,historia-del-narcotrafico-en-mexico,Historia del narcotráfico en México,"Historia del narcotráfico en México es, sin du..."
1495,mama-come-sano-alimentacion-saludable-en-el-em...,Mamá come sano: Alimentación saludable en el e...,El embarazo y la lactancia son etapas maravill...
1533,el-testamento-de-san-juan,El Testamento de San Juan,Ciudad de Éfeso. Año sexto del gobierno del em...
1536,palabra-y-vida-2013-el-evangelio-comentado-cad...,Palabra y Vida 2013 – El Evangelio comentado c...,Palabra y Vida 2013 nos invita este año \nespe...


In [205]:
len(filas_a_normalizar)

45

In [206]:
libros[columnas_texto] = libros[columnas_texto].apply(
    lambda columna: columna.str.replace(r'\s+', ' ', regex=True).str.strip()
)


In [207]:
# Filtrar el DataFrame para obtener todos los registros con sinopsis repetida
lineas_repetidas = libros[libros['synopsis'].duplicated(keep=False)].sort_values('synopsis')

# Mostrar las columnas más relevantes para comparar los duplicados
lineas_repetidas[['slug', 'title', 'authors', 'synopsis']].head(10)

,slug,title,authors,synopsis
7911,practicas-sexuales-alternativas-i,Prácticas sexuales alternativas (I),[César Martín],Al repasar la larga lista de prácticas sexuale...
7912,practicas-sexuales-alternativas-ii,Prácticas sexuales alternativas (II),[César Martín],Al repasar la larga lista de prácticas sexuale...
7583,la-historia-interminable,La Historia Interminable,[Michael Ende],"Bastián Baltasar Bux, es un niño tímido al que..."
7582,la-historia-interminable-color,La Historia Interminable – Color,[Michael Ende],"Bastián Baltasar Bux, es un niño tímido al que..."
6574,la-historia-de-colditz,La historia de Colditz,[Patrick R. Reid],"Colditz, un castillo-fortaleza convertido en p..."
6575,ultimos-dias-en-colditz,Últimos días en Colditz,[Patrick R. Reid],"Colditz, un castillo-fortaleza convertido en p..."
3455,los-anos-irreverentes-escritos-completos-en-el...,Los años irreverentes. Escritos completos en E...,"[Flavia de la Fuente, Quintín]","Como todas las cosas que suceden en la vida, h..."
3456,los-anos-irreverentes-escritos-completos-en-el...,Los años irreverentes. Escritos completos en E...,"[Flavia de la Fuente, Quintín]","Como todas las cosas que suceden en la vida, h..."
4558,comunicaciones-unificadas-con-elastix,Comunicaciones unificadas con Elastix,[Edgar Landívar],Comunicaciones Unificadas con Elastix es una o...
4570,comunicaciones-unificadas-con-elastix-volumen-2,Comunicaciones unificadas con Elastix (Volumen 2),[Edgar Landívar],Comunicaciones Unificadas con Elastix es una o...


In [208]:
# Agrupa por sinopsis y cuanta las ocurrencias de cada sinopsis
conteo_sinopsis = libros.groupby('synopsis').size().reset_index(name='cantidad')

# Filtra solo los textos que aparecen más de 1 vez y ordenarlos de mayor a menor
sinopsis_repetidas = conteo_sinopsis[conteo_sinopsis['cantidad'] > 1].sort_values(by='cantidad', ascending=False)

# Mostrar el texto de la sinopsis y la cantidad de repeticiones
sinopsis_repetidas

,synopsis,cantidad
1740,El Curso de Alemán consta de 96 Unidades didác...,58
6434,Revista española de ciencia ficción y fantasía...,7
2440,El «Teatro crítico universal» (teatro ha de en...,4
5922,"Nono (siglo V d. C.), natural de Panópolis, en...",3
1857,"El autor, llamado Kara Ben Nemsi (Carlos, hijo...",3
305,Al repasar la larga lista de prácticas sexuale...,2
866,"Como todas las cosas que suceden en la vida, h...",2
879,Comunicaciones Unificadas con Elastix es una o...,2
598,"Bastián Baltasar Bux, es un niño tímido al que...",2
2058,El gran friso narrativo de los Episodios Nacio...,2


In [209]:
# Eliminar duplicados exactos conservando solo el primero
libros = libros.drop_duplicates(subset=['synopsis'], keep='first')

# Calcula cuántos se eliminaron
eliminados_sinopsis = cantidad_total - len(libros)
eliminados['sinopsis duplicada'] = eliminados_sinopsis

print(f"Se eliminaron {eliminados_sinopsis} libros por tener sinopsis exacta duplicada.")
print(f"Total de libros restantes: {len(libros)}")

Se eliminaron 98 libros por tener sinopsis exacta duplicada.
Total de libros restantes: 8648


In [210]:
# Crea una columna temporal de autor en formato string para poder agrupar
libros['autor_str'] = libros['authors'].apply(lambda x: ", ".join(x) if isinstance(x, list) else str(x))

libros_a_borrar = set()
pares_unidos = []
pares_separados = []

# Itera solo por los autores que tengan más de un libro
for autor, df_autor in libros.groupby('autor_str'):
    if len(df_autor) > 1:
        indices = df_autor.index.tolist()
        
        # Compara todos contra todos dentro del mismo autor
        for i in range(len(indices)):
            for j in range(i + 1, len(indices)):
                idx1, idx2 = indices[i], indices[j]
                
                # Ignora si alguno ya fue marcado para eliminación
                if idx1 in libros_a_borrar or idx2 in libros_a_borrar:
                    continue
                
                tit1 = str(df_autor.loc[idx1, 'title']).lower()
                tit2 = str(df_autor.loc[idx2, 'title']).lower()
                
                sim_titulo = fuzz.token_sort_ratio(tit1, tit2)
                
                # CRITERIO: Si los títulos se parecen mucho (>85%)
                if sim_titulo > 85:
                    sinop1 = str(df_autor.loc[idx1, 'synopsis']).lower()
                    sinop2 = str(df_autor.loc[idx2, 'synopsis']).lower()
                    
                    sim_sinopsis = fuzz.token_set_ratio(sinop1, sinop2)
                    
                    # Si la sinopsis también coincide (>75%), es la misma obra
                    if sim_sinopsis > 75:
                        libros_a_borrar.add(idx2) # Marcamos la segunda versión para borrar
                        if len(pares_unidos) < 2:
                            pares_unidos.append((tit1, tit2))
                    
                    # Si no coincide la sinopsis, son libros distintos (Ej: Vol 1 y Vol 2)
                    else:
                        if len(pares_separados) < 2:
                            pares_separados.append((tit1, tit2))

# Elimina las copias casi exactas y la columna auxiliar
libros = libros.drop(index=list(libros_a_borrar))
libros = libros.drop(columns=['autor_str'])

# Actualiza el diccionario de eliminados
eliminados['similitud de cadenas'] = len(libros_a_borrar)

# Muestra los resultados
print("--- CRITERIO APLICADO ---")
print("Títulos similares > 85% + Sinopsis similares > 75% -> Mismo libro (Unir)")
print("Títulos similares > 85% + Sinopsis distintas <= 75% -> Distinto libro (Separar)\n")

print("--- DOS PARES UNIDOS (Misma obra) ---")
for p in pares_unidos: print(f"- {p[0]}  |  {p[1]}")

print("\n--- DOS PARES SEPARADOS (Distinta obra) ---")
for p in pares_separados: print(f"- {p[0]}  |  {p[1]}")

print("\n--- RESUMEN FINAL PASO 1 ---")
print(f"Motivos de eliminación: {eliminados}")
print(f"Libros que quedaron disponibles: {len(libros)}\n")

# 5 registros finales
libros.head(5)

--- CRITERIO APLICADO ---
Títulos similares > 85% + Sinopsis similares > 75% -> Mismo libro (Unir)
Títulos similares > 85% + Sinopsis distintas <= 75% -> Distinto libro (Separar)

--- DOS PARES UNIDOS (Misma obra) ---
- la perla número 8  |  la perla número 5
- la perla número 8  |  la perla número 6

--- DOS PARES SEPARADOS (Distinta obra) ---
- nuevos cuentos de bustos domecq  |  nuevos cuentos de bustos domecq
- historias negras del fútbol argentino  |  nuevas historias negras del fútbol argentino

--- RESUMEN FINAL PASO 1 ---
Motivos de eliminación: {'sin sinopsis': 1, 'slug duplicado': 0, 'sinopsis duplicada': 98, 'similitud de cadenas': 43}
Libros que quedaron disponibles: 8605



,slug,title,authors,synopsis
0,de-la-caverna-al-cosmos,De la caverna al cosmos,[Eudald Carbonell],La Humanidad ha entrado en una nueva fase evol...
1,quienes-somos-y-como-hemos-llegado-hasta-aqui,Quiénes somos y cómo hemos llegado hasta aquí,[David Reich],La extracción de ADN de los huesos de nuestros...
2,teotihuacan,Teotihuacan,[Eduardo Matos Moctezuma],Teotihuacan es la primera gran ciudad del cent...
3,tiro-y-las-colonias-fenicias-de-occidente,Tiro y las colonias fenicias de Occidente,[M.ª Eugenia Aubet],Entre los siglos VIII y VI a. C. los fenicios ...
4,la-maldicion-del-hombre-mono,La maldición del hombre mono,[Emiliano Bruner],Las raíces evolutivas del sufrimiento humano U...
